# Adım 6 — Context Adjust Engine

Adım 5'in ürettiği `fraud_score`'u **iş bağlamına göre** ince ayar yapar.  
Girdi: `df_final_train.parquet` (590,540 × 510)  
Çıktı: `df_adjusted_train.parquet` + `context_report_train.json`

---

## Tasarım Felsefesi

> **"Skor silmez, düzeltir."**

Bu adım bir filtre değil, bir **bağlam katmanı**. Her katman `multiplicative adjustment` uygular:

```
context_adjusted_score = fraud_score × composite_multiplier   →   [0, 1] clip
composite_multiplier   = katman₁ × katman₂ × katman₃ × katman₄ × katman₅
```

**5 Context Katmanı:**

| # | Katman | Mantık | Etki |
|---|--------|--------|------|
| 1 | Business Hours | Mesai içi → false positive azalt | 0.85 / 1.00 / 1.15 |
| 2 | Weekend | Hafta sonu + yüksek tutar → risk artır | 0.95 / 1.00 / 1.10 |
| 3 | Trusted Entity | Düşük velocity + stabil tutar → güvenilir | 0.85 / 1.00 |
| 4 | Transaction Type | ProductCD fraud lift'e göre | 0.90 / 1.00 / 1.20 |
| 5 | High Value Night | Gece + 95. percentil üzeri tutar → agresif | 1.00 / 1.30 |

**Çalışma sonucu:**
```
Ayrım oranı önce : 2.0092x
Ayrım oranı sonra: 2.0914x   (+0.0822)
```

## Fonksiyonlar

### 1. `apply_business_hours_context(df)`

İşlem saatine göre risk çarpanı uygular.

```
08:00 – 18:00  →  0.85   (mesai saati: gözetim yüksek, false positive azalt)
18:00 – 22:00  →  1.00   (akşam: nötr)
22:00 – 08:00  →  1.15   (gece: düşük gözetim, risk artır)
```

> **Etki:** 590,540 işlemin %71.7'sini etkiliyor.  
> 233,465 işlem ↑ (gece), 189,898 işlem ↓ (mesai içi)

### 2. `apply_weekend_context(df)`

Hafta sonu + tutar kombinasyonuna göre multiplier üretir.

```
Hafta içi                          →  1.00
Hafta sonu + tutar ≤ medyan        →  0.95   (rutin alışveriş)
Hafta sonu + tutar > medyan        →  1.10   (yüksek tutarlı hafta sonu)
```

> **Etki:** %28.8 işlem etkileniyor.  
> Medyan eşiği train veriden otomatik hesaplanır — statik değer yok.

### 3. `apply_trusted_entity_context(df)`

Davranışsal tutarlılık gösteren entity'lerin skorunu düşürür.

**Kriter — her ikisi de sağlanmalı (AND):**
- `card1_velocity_1d ≤ 6` → düşük günlük işlem hızı
- `card1_amt_zscore ≤ 0.5` → tutarı tarihsel ortalamasına yakın

```
Her iki kriter sağlanır  →  0.85   (güvenilir, false positive azalt)
Sağlanmaz                →  1.00
```

> **Tasarım notu:** `card1_tx_count` güvenilmez proxy olarak reddedildi.  
> (min=1, max=14,932, medyan=919 — kart frekansını sayıyor, davranışı değil)  
> Bunun yerine davranışsal kolonlara geçildi: velocity + zscore.

### 4. `apply_transaction_type_context(df)`

ProductCD'nin `fraud_lift` değerine göre ürün bazlı risk uygular.

```
lift < 0.80   →  0.90   (düşük riskli ürün)
0.80 ≤ lift ≤ 1.20  →  1.00   (nötr)
lift > 1.20   →  1.20   (yüksek riskli ürün)
```

> **Veri odaklı:** Statik ProductCD → risk eşlemesi yok.  
> Lift değeri doğrudan Adım 3'ün ürettiği `ctx_productcd_fraud_lift` kolonundan gelir.  
> **Etki:** %93.6 işlem etkileniyor — en geniş kapsam. 439,670 işlem ↓ (lift değerleri yoğunlukla 0.583'te kümeleniyor)

### 5. `apply_high_value_night_context(df)`

Gece saati + aşırı yüksek tutar kombinasyonu için özel katman.

```
Gece (22:00–06:00) VE tutar > 95. percentil  →  1.30
Aksi                                          →  1.00
```

**Business Hours ile kümülatif etki:**
```
Gece işlemi:          ×1.15   (business hours katmanı)
Gece + yüksek tutar:  ×1.15 × ×1.30 = ×1.495
```

> **Etki:** Sadece %1.4 işlem etkileniyor — hedefli, agresif artış.  
> 95. percentil eşiği train veriden otomatik hesaplanır.

### 6. `run_context_adjustment()` — Orkestratör

Tüm katmanları sırayla çalıştırır, raporlar.

**Akış:**
```
1. Her katman → kendi multiplier kolonunu üretir
2. composite_multiplier = tüm katmanların çarpımı
3. context_adjusted_score = fraud_score × composite_multiplier → [0,1] clip
4. context_report üretilir (etkilenen satır sayıları, separation before/after)
```

**Üretilen yeni kolonlar:**
`business_hours_multiplier`, `weekend_multiplier`, `trusted_entity_multiplier`,  
`product_type_multiplier`, `high_value_night_multiplier`,  
`composite_multiplier`, `context_adjusted_score`

In [1]:
# ============================================================
# ADIM 6 — Context Adjust Engine
# ============================================================
# Bağımlılıklar: Adım 5 çıktısı (df_final_train.parquet)
# Girdi kolonları:
#   tx_hour, tx_is_weekend, TransactionAmt, card1_tx_count,
#   amt_zscore, ctx_productcd_fraud_lift, ProductCD
# Çıktı: df_adjusted + context_report
# ============================================================

import pandas as pd
import numpy as np
import json
from pathlib import Path

# ── Sabitler ────────────────────────────────────────────────
SCORE_COL          = "fraud_score"
TARGET_COL         = "isFraud"
OUTPUT_DIR         = Path("outputs/step6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. apply_business_hours_context
# ============================================================
def apply_business_hours_context(
    df: pd.DataFrame,
    tx_hour_col: str = "tx_hour",
) -> pd.Series:
    """
    Saate göre risk multiplier üretir.

    Mantık:
        08-18  → mesai içi   → 0.85 (düşür, false positive azalt)
        18-22  → akşam       → 1.00 (dokunma)
        22-08  → gece        → 1.15 (artır)

    Parametre olarak dışarıdan override edilebilir.
    """
    hour = df[tx_hour_col]

    multiplier = pd.Series(1.00, index=df.index, name="business_hours_multiplier")
    multiplier = multiplier.where(~((hour >= 8) & (hour < 18)), other=0.85)
    multiplier = multiplier.where(~((hour >= 22) | (hour < 8)), other=1.15)

    return multiplier


# ============================================================
# 2. apply_weekend_context
# ============================================================
def apply_weekend_context(
    df: pd.DataFrame,
    weekend_col: str  = "tx_is_weekend",
    amt_col: str      = "TransactionAmt",
    amt_threshold: float = None,   # None → train medyanı hesaplanır
) -> pd.Series:
    """
    Hafta sonu + tutar kombinasyonuna göre multiplier üretir.

    Mantık:
        Hafta içi                        → 1.00
        Hafta sonu + tutar <= medyan     → 0.95 (rutin alışveriş)
        Hafta sonu + tutar >  medyan     → 1.10 (yüksek tutarlı hafta sonu)
    """
    if amt_threshold is None:
        amt_threshold = df[amt_col].median()

    is_weekend   = df[weekend_col] == 1
    is_high_amt  = df[amt_col] > amt_threshold

    multiplier = pd.Series(1.00, index=df.index, name="weekend_multiplier")
    multiplier = multiplier.where(~(is_weekend & ~is_high_amt), other=0.95)
    multiplier = multiplier.where(~(is_weekend &  is_high_amt), other=1.10)

    return multiplier


# ============================================================
# 3. apply_trusted_entity_context
# ============================================================
def apply_trusted_entity_context(
    df: pd.DataFrame,
    velocity_col: str     = "card1_velocity_1d",
    amt_zscore_col: str   = "card1_amt_zscore",
    max_velocity: float   = 3.0,
    max_zscore: float     = 0.3,
) -> pd.Series:
    is_low_velocity = df[velocity_col] <= max_velocity
    is_stable_amt   = df[amt_zscore_col].abs() <= max_zscore

    multiplier = pd.Series(1.00, index=df.index, name="trusted_entity_multiplier")
    # Sadece ikisi birden sağlanırsa indirim
    multiplier = multiplier.where(~(is_low_velocity & is_stable_amt), other=0.85)

    return multiplier


# ============================================================
# 4. apply_transaction_type_context
# ============================================================
def apply_transaction_type_context(
    df: pd.DataFrame,
    fraud_lift_col: str = "ctx_productcd_fraud_lift",
    lift_low: float     = 0.8,
    lift_high: float    = 1.2,
    mult_low: float     = 0.90,
    mult_high: float    = 1.20,
) -> pd.Series:
    """
    ctx_productcd_fraud_lift kolonunu kullanarak product bazlı risk uygular.

    Lift değeri veriden gelir (Adım 3 context features).
    Statik map yok — veri odaklı.

    Mantık:
        lift < lift_low  → düşük riskli ürün tipi → 0.90
        lift > lift_high → yüksek riskli ürün tipi → 1.20
        arası            → nötr                   → 1.00
    """
    lift = df[fraud_lift_col]

    multiplier = pd.Series(1.00, index=df.index, name="product_type_multiplier")
    multiplier = multiplier.where(~(lift < lift_low),  other=mult_low)
    multiplier = multiplier.where(~(lift > lift_high), other=mult_high)

    return multiplier


# ============================================================
# 5. apply_high_value_night_context
# ============================================================
def apply_high_value_night_context(
    df: pd.DataFrame,
    tx_hour_col: str    = "tx_hour",
    amt_col: str        = "TransactionAmt",
    night_start: int    = 22,
    night_end: int      = 6,
    amt_percentile: float = 95.0,
    multiplier_val: float = 1.30,
) -> pd.Series:
    """
    Gece saati + aşırı yüksek tutar kombinasyonu → agresif artış.

    Bu katman business_hours'tan ayrı çalışır:
    business_hours tüm gece işlemlerini 1.15 ile çarpar.
    Bu katman sadece gece + 95. percentil üzeri tutarlara odaklanır.
    İki multiplier çarpılınca etki kümülatif olur (1.15 × 1.30 = 1.495).
    """
    amt_threshold = np.percentile(df[amt_col].dropna(), amt_percentile)

    hour      = df[tx_hour_col]
    is_night  = (hour >= night_start) | (hour < night_end)
    is_high   = df[amt_col] > amt_threshold

    multiplier = pd.Series(1.00, index=df.index, name="high_value_night_multiplier")
    multiplier = multiplier.where(~(is_night & is_high), other=multiplier_val)

    return multiplier


# ============================================================
# 6. run_context_adjustment — Orkestratör
# ============================================================
def run_context_adjustment(
    df: pd.DataFrame,
    score_col: str           = SCORE_COL,
    target_col: str          = TARGET_COL,
    tx_hour_col: str         = "tx_hour",
    weekend_col: str         = "tx_is_weekend",
    amt_col: str             = "TransactionAmt",
    velocity_col: str        = "card1_velocity_1d",   # güncellendi
    amt_zscore_col: str      = "card1_amt_zscore",    # güncellendi
    fraud_lift_col: str      = "ctx_productcd_fraud_lift",
    max_velocity: float      = 6.0,                   # yeni
    max_zscore: float        = 0.5,                   # yeni
    amt_percentile: float    = 95.0,
    lift_low: float          = 0.80,
    lift_high: float         = 1.20,
) -> tuple[pd.DataFrame, dict]:
    """
    Tüm context katmanlarını sıraya dizer, çarpar, raporlar.

    Adım:
        1. Her katman kendi multiplier kolonunu üretir
        2. composite_multiplier = tüm katmanların çarpımı
        3. context_adjusted_score = score × composite → [0,1] clip
        4. context_report üretilir

    Dönüş:
        df_adjusted   : orijinal df + multiplier kolonları + adjusted score
        context_report: etki özeti (json olarak kaydedilir)
    """
    df = df.copy()


    # ── Multiplier kolonları ─────────────────────────────────
    df["business_hours_multiplier"] = apply_business_hours_context(
        df, tx_hour_col=tx_hour_col
    )
    df["weekend_multiplier"] = apply_weekend_context(
        df, weekend_col=weekend_col, amt_col=amt_col
    )
    df["trusted_entity_multiplier"] = apply_trusted_entity_context(
        df,
        velocity_col=velocity_col,
        amt_zscore_col=amt_zscore_col,
        max_velocity=max_velocity,
        max_zscore=max_zscore,
    )
    df["product_type_multiplier"] = apply_transaction_type_context(
        df, fraud_lift_col=fraud_lift_col, lift_low=lift_low, lift_high=lift_high
    )
    df["high_value_night_multiplier"] = apply_high_value_night_context(
        df, tx_hour_col=tx_hour_col, amt_col=amt_col, amt_percentile=amt_percentile
    )

    # ── Composite & adjusted score ───────────────────────────
    multiplier_cols = [
        "business_hours_multiplier",
        "weekend_multiplier",
        "trusted_entity_multiplier",
        "product_type_multiplier",
        "high_value_night_multiplier",
    ]
    df["composite_multiplier"]      = df[multiplier_cols].prod(axis=1)
    df["context_adjusted_score"]    = (
        df[score_col] * df["composite_multiplier"]
    ).clip(0, 1)

    # ── Context Report ───────────────────────────────────────
    total = len(df)
    report = {"total_rows": total, "layers": {}}

    for col in multiplier_cols:
        affected = int((df[col] != 1.00).sum())
        pushed_up   = int((df[col] > 1.00).sum())
        pushed_down = int((df[col] < 1.00).sum())
        report["layers"][col] = {
            "affected_rows" : affected,
            "affected_pct"  : round(affected / total * 100, 2),
            "pushed_up"     : pushed_up,
            "pushed_down"   : pushed_down,
            "mean_multiplier": round(float(df[col].mean()), 4),
        }

    # Ayrım oranı: önce vs sonra
    def separation_ratio(scores, labels):
        fraud_mean     = scores[labels == 1].mean()
        nonfraud_mean  = scores[labels == 0].mean()
        return round(float(fraud_mean / nonfraud_mean), 4) if nonfraud_mean > 0 else None

    if target_col in df.columns:
        report["separation_before"] = separation_ratio(df[score_col],              df[target_col])
        report["separation_after"]  = separation_ratio(df["context_adjusted_score"], df[target_col])
        report["separation_delta"]  = round(
            report["separation_after"] - report["separation_before"], 4
        )

        # Fraud / non-fraud ortalama skor
        for col in [score_col, "context_adjusted_score"]:
            report[f"{col}_fraud_mean"]    = round(float(df.loc[df[target_col]==1, col].mean()), 4)
            report[f"{col}_nonfraud_mean"] = round(float(df.loc[df[target_col]==0, col].mean()), 4)

    # Composite multiplier dağılımı
    report["composite_multiplier_stats"] = {
        "mean"  : round(float(df["composite_multiplier"].mean()), 4),
        "min"   : round(float(df["composite_multiplier"].min()),  4),
        "max"   : round(float(df["composite_multiplier"].max()),  4),
        "median": round(float(df["composite_multiplier"].median()),4),
    }

    return df, report




## Çalıştır

**Girdi:** `step5/df_final_train.parquet`  
**Parametreler:** `lift_low=0.80`, `lift_high=1.20`, `max_velocity=6.0`, `max_zscore=0.5`

Beklenen katman etkileri:
```
business_hours_multiplier    → etkilenen: %71.7   ↑233,465  ↓189,898
weekend_multiplier           → etkilenen: %28.8   ↑85,443   ↓84,728
trusted_entity_multiplier    → etkilenen: %29.5   ↑0        ↓174,023
product_type_multiplier      → etkilenen: %93.6   ↑113,171  ↓439,670
high_value_night_multiplier  → etkilenen:  %1.4   ↑8,153    ↓0

Composite multiplier: mean=0.94, min=0.62, max=1.97
Ayrım oranı: 2.0092x → 2.0914x  (+0.0822)
```

In [2]:
# ============================================================
# ÇALIŞTIRMA
# ============================================================
if __name__ == "__main__":

    print("Adım 6 — Context Adjust Engine")
    print("=" * 50)

    # Adım 5 çıktısını yükle
    df_input = pd.read_parquet("../Adım 5/outputs/step5/df_final_train.parquet")
    print(f"Yüklendi: {df_input.shape}")

    # Context adjustment uygula
    df_adjusted, context_report = run_context_adjustment(
        df_input,
        lift_low=0.80,
        lift_high=1.20,
        max_velocity=6.0,
        max_zscore=0.5,
    )

    # Raporu yazdır
    print("\n── Context Report ──────────────────────────────")
    print(f"Ayrım oranı önce : {context_report.get('separation_before')}")
    print(f"Ayrım oranı sonra: {context_report.get('separation_after')}")
    print(f"Delta            : {context_report.get('separation_delta')}")

    print("\n── Katman Etkileri ─────────────────────────────")
    for layer, stats in context_report["layers"].items():
        print(f"{layer:35s} → etkilenen: {stats['affected_pct']:5.1f}%  "
              f"↑{stats['pushed_up']}  ↓{stats['pushed_down']}")

    print("\n── Composite Multiplier ────────────────────────")
    for k, v in context_report["composite_multiplier_stats"].items():
        print(f"  {k}: {v}")

    # Kaydet
    df_adjusted.to_parquet(OUTPUT_DIR / "df_adjusted_train.parquet", index=False)
    with open(OUTPUT_DIR / "context_report_train.json", "w") as f:
        json.dump(context_report, f, indent=2)

    print(f"\nKaydedildi → {OUTPUT_DIR}")

Adım 6 — Context Adjust Engine
Yüklendi: (590540, 510)

── Context Report ──────────────────────────────
Ayrım oranı önce : 2.0092
Ayrım oranı sonra: 2.0914
Delta            : 0.0822

── Katman Etkileri ─────────────────────────────
business_hours_multiplier           → etkilenen:  71.7%  ↑233465  ↓189898
weekend_multiplier                  → etkilenen:  28.8%  ↑85443  ↓84728
trusted_entity_multiplier           → etkilenen:  29.5%  ↑0  ↓174023
product_type_multiplier             → etkilenen:  93.6%  ↑113171  ↓439670
high_value_night_multiplier         → etkilenen:   1.4%  ↑8153  ↓0

── Composite Multiplier ────────────────────────
  mean: 0.9436
  min: 0.6177
  max: 1.9734
  median: 0.9

Kaydedildi → outputs\step6


---

## Sonuç Yorumu

### Separation İyileşmesi

```
Adım 5 çıktısı        : 2.0092x
Adım 6 sonrası        : 2.0914x   (+0.082)
```

Context katmanı separation'ı anlamlı artırdı ama hedef **2.3x**'e ulaşılamadı.  
Bunun temel nedeni: `trusted_entity_multiplier` fraud/non-fraud ayrımında yetersiz kaldı  
(fraud: 0.938 vs non-fraud: 0.927 — neredeyse aynı).

### Neden Hedef Aşılamadı?

`ctx_productcd_fraud_lift` değerlerinin **%75'i 0.583'te kümeleniyor** —  
bu değerler `lift_low=0.80` eşiğinin altında kaldığı için neredeyse tüm işlemler `×0.90` aldı.  
Bu da non-fraud skorları geniş ölçekte düşürdü, fraud/non-fraud ayrımını sınırladı.

### Çözüm: Adım 7 Rule Engine

Daha güçlü sinyaller kullanılacak:
- `ctx_card4_fraud_lift`, `ctx_card6_fraud_lift` — kart tipi bazlı lift
- Gece + velocity + email domain kombinasyon kuralları

**Sonraki adım:** Adım 7 — Rule Engine  
Girdi: `df_adjusted_train.parquet`  
